# WASP Relational Learning Course Project

In this project, we are going to re-implement classifiers for unsolvable classical planning states in the Clingo Answer Set Programming (ASP) language based on the paper by Stahlberg et al.

Link to the paper: [**Learning Generalized Unsolvability Heuristics for Classical Planning**](https://mrlab.ai/papers/stahlberg-et-al-ijcai2021.pdf)

1. The problem setting is as follows:

We are given a **class of classical planning tasks** $Q$, consisting of a set of **planning tasks** $\Pi = (I, D)$ where $D$ is a common first-order **planning domain**, consisting of **predicates** and **action schemas**, and $I$ is task-specific information, consisting of **objects**, and two sets of ground atoms, one specifying the **initial situation** and another one specifying a **goal situation**. Each $\Pi$ induces a **state model**, where ground actions, i.e., actions where all variables are substituted by objects, form transitions from one situation to another, also called **states**. In other words, a state model is a [transition system](https://en.wikipedia.org/wiki/Transition_system) and a path from the node representing the initial situation to a node representing a goal situation is called a **plan**. Search algorithms such as A* or best-first search with guidance based on heuristic are one approach of finding plans. We say that a node in the transition system is **unsolvable**, if there exists no path from it to a node representing a goal situation. Conversely, a state is **solvable** iff it is not unsolvable. 

2. Motivation for Unsolvability Heuristics:

Detecting whether a node is unsolvable can greatly improve search performance, as it allows pruning a potentially huge number of states. Hence, **the objective in this project** is to write an algorithm for learning to separate solvable from unsolvable states in any problem $\Pi$ in $Q$. Since $Q$ may be infinite, we focus on learning a classifier from small planning tasks, but that generalizes towards tasks of any given size, where size is measured by the number of objects.

3. Representation Language

There are different ways to design such a classifier. Given the objective of learning from small tasks, measured by the number of objects in the task, while generalizing towards arbitrary sized tasks, we are limited to using languages that use information that is common among the planning tasks in $Q$, namely the planning domain $D$. The language is described in more detail in the paper. Given this language, we can write features that evaluate to true of false in any state in any problem $\Pi$ in $Q$ whose meaning is logically characterized. For example, in a Logistics problem, there may be a feature that counts the number of undelivered packages, in any problem $\Pi$ in $Q$. In this notebook, it is fine to view those features as black boxes, while viewing them as Boolean state features.

The notebook is structured as follows:

- Step 0: Install required packages,
- Step 1: Inspect the data, gloss the features, and record a hypothesis about which features will matter,
- Step 2: Create train/valid/test splits,
- Step 3: Train a baseline ML classifier (KNN) on the same split for reference,
- Step 4: Learn a T-Perfect generalized unsolvability heuristic in ASP,
- Step 5: Learn a T-Safe generalized unsolvability heuristic in ASP, reusing most of the T-Perfect code,
- Step 6: Open-ended bake-off against standard ML classifiers (decision tree required; others optional).



# Step 0: Install Required Packages

In [2]:
!uv pip install clingo pandas scikit-learn


Checked 3 packages in 502ms


# Step 1: Inspect the data

Column labels: syntactic representation of the features
Row labels: syntactic feature complexities (row 0), state indices (row 1,2,...)

Consider row-major data representation. Then

data[0][i] is the syntactic feature complexity of the feature f_i.
data[j][i] with j=1,2,... is the value of the feature f_i in state s_j.

In [3]:
import pandas as pd

# read the table
df = pd.read_csv("train-features/spanner/features_small.csv")
df.head()

,(> |[Exists at [Exists link [ForAll link [ForAll [Inverse at] spanner]]]]| |[Exists carrying useable]|),(> |[Not loose]| |[And [ForAll carrying tightened] [ForAll at [Exists [Inverse at] man]]]|),(> |useable| |[Exists link [Exists [Transitive link] [Exists link [Exists [Inverse at] loose]]]]|),(> |[And useable [ForAll at [ForAll link [Exists [Inverse at] spanner]]]]| |nut|),(> |[And [Not [Or man tightened]] [RoleValue [Inverse at] [Inverse carrying]]]| |[RoleValue link carrying]|),(> |[Exists at [ForAll [Inverse link] [Exists [Inverse at] All]]]| |loose|),(> |[Exists at [Exists link [ForAll [Inverse at] man]]]| |[Exists carrying locatable]|),(> |[Exists at [And [ForAll [Inverse at] [Not tightened]] [ForAll link [RoleValue link [Transitive link]]]]]| 0),(> |[Or loose [Exists at [Exists [Transitive link] [Exists [Inverse at] man]]]]| |spanner|),(== |nut| |[And [ForAll [Inverse at] [Not spanner]] [ForAll at [ForAll [Inverse at] man]]]|),...,(> |[And locatable [ForAll at [Exists link [And [ForAll link nut] [ForAll [Inverse at] nut]]]]]| 0),(> |nut| |[Exists [Transitive link] [Exists [Inverse at] [Or spanner [Exists carrying locatable]]]]|),(== |[Exists at [Exists link [Exists link [ForAll link [Exists [Inverse at] [Exists carrying useable]]]]]]| 0),(> |[Or loose useable]| |[And [ForAll link nut] [ForAll [Inverse carrying] [ForAll carrying useable]]]|),(== |[Exists link [Exists [Inverse at] All]]| |[Not [ForAll at [Exists [Inverse at] tightened]]]|),(> |[And [Or useable [Not spanner]] [ForAll at [ForAll [Inverse at] man]]]| |locatable|),(== |location| |[Or loose [Or [Exists [Inverse at] man] [Exists link [ForAll link nut]]]]|),(== |nut| |[And [Or man spanner] [ForAll at [ForAll link [ForAll [Inverse at] man]]]]|),dead-end,source
0,14,13,13,12,14,11,12,15,13,14,...,15,13,15,14,14,13,14,14,-1,-1
1,1,1,0,0,1,0,1,1,1,0,...,0,1,1,0,0,0,0,1,1,p11.out.txt
2,1,0,1,0,0,1,0,1,1,0,...,1,0,1,0,0,0,0,1,1,p15.out.txt
3,1,0,1,0,0,1,0,1,1,0,...,1,0,1,0,0,0,0,0,1,p17.out.txt
4,0,0,1,0,0,1,0,1,1,0,...,1,0,1,0,0,0,0,1,1,p18.out.txt


### Feature Glossing

Column names in this dataset are written in the feature representation language from
Stahlberg et al. (2021). Each name encodes a logical formula over the planning state;
interpreting it requires reading the *structure* of the formula, not just recognising
the predicate names.

**Formula grammar (brief):**
- `|p|` — count of objects satisfying unary predicate `p`
- `|[Exists r p]|` — count of objects reachable via role `r` that satisfy `p`
- `|[ForAll r p]|` — count of objects where *all* role-`r` successors satisfy `p`
- `|[And p q]|`, `|[Or p q]|`, `|[Not p]|` — Boolean combinations
- `|[Inverse r]|` — reverse direction of role `r`
- `(> A B)`, `(== A B)` — numeric comparison; feature value is 1 (true) or 0 (false)

**Worked example:**

Raw name:
`(> |[Exists at [Exists link [ForAll [Inverse at] man]]]| |[Exists carrying locatable]|)`

Sample gloss: *"True when the count of objects located at a spot that links to a place
occupied only by the man exceeds the count of objects carrying a locatable thing."*

The gloss is a literal translation of the formula structure, nothing more — read each
bracketed sub-formula inside out, then assemble the comparison.

---

**Your task:** Write a one-sentence plain-English gloss for each feature column.
If that is too many, gloss at minimum the **five features with the lowest complexity
score** (complexity is row 0 of the dataframe).

**Your answer:**

--- Top 5 Lowest Complexity Features ---
Rank 1 | Column Index: 39 | Complexity: 10
Formula: (> |locatable| |[ForAll at [Exists link [Exists [Inverse at] tightened]]]|)
Gloss: True when the count of locatable objects exceeds the count of objects whose locations all links to a location that has a tightened object.

Rank 2 | Column Index: 5 | Complexity: 11
Formula: (> |[Exists at [ForAll [Inverse link] [Exists [Inverse at] All]]]| |loose|)
Gloss: True when the count of objects located at a spot whose neighbouring objects are all occupied by some objects exceeds the count of loose objects.

Rank 3 | Column Index: 12 | Complexity: 11
Formula: (== |spanner| |[And locatable [ForAll at [ForAll link [Exists link location]]]]|)
Gloss: True when the count of spanners equals the count of locatable objects whose locations are all links lead to a spot that has a link. 

Rank 4 | Column Index: 11 | Complexity: 11
Formula: (> |useable| |[Not [ForAll link [ForAll link [Exists [Inverse at] man]]]]|)
Gloss: True when the count of useable objects exceeds the count of objects that are not at the spot from which all directions lead to a spot occupied by man. 

Rank 5 | Column Index: 38 | Complexity: 11
Formula: (> |[Exists at [ForAll [Inverse at] nut]]| |[Exists [Inverse at] spanner]|)
Gloss: True when the count of objects located at a spot by object exceeds the count of objects at a spot that has a spanner. 

### Hypothesis: Which Features Matter?

Before training any classifier, record your prediction here.

Which **two or three features** do you expect to be the most predictive of dead-end
states, and why? Refer to the glosses you wrote above, or to your reading of the feature
formulas directly.

You will revisit this prediction after Step 4 to see whether the ASP solver agrees.

**Your answer:** 
From the gloss captured above with lowest complexity, I pick last two formulas, 

Rank 4 | Column Index: 11 | Complexity: 11
Formula: (> |useable| |[Not [ForAll link [ForAll link [Exists [Inverse at] man]]]]|)
Reason: This feature tracks irreversible movement. Here, the man can only walk forward along the path links that he can never go back. If he walks pasing a room that contains tools without picking them up, those tools are lost forever. The gloss captures the number of usable spanners left is greater than the locations man has already passed. Therefore, ending in a dead-end state.

Rank 5 | Column Index: 38 | Complexity: 11
Formula: (> |[Exists at [ForAll [Inverse at] nut]]| |[Exists [Inverse at] spanner]|)
Reason: This feature tracks the resource bottlenecks. It directly compares the number of "goal targets" (rooms where nuts need to be fixed) against the number of "tool rooms" (rooms where spanner is available to be grabbed). Here, the number of rooms that requires work outnumbers that room that provide the tootls to perform the work, the man runs out of resources before satisfying the goals. Therefore, ending in a dead-end state. 

## Step 2: Create a Train/Valid/Test Split

In [4]:
from sklearn.model_selection import train_test_split

# keep complexity row separately
df_features = df.iloc[0].drop(labels=["dead-end", "source"])
# Actual data states from row 1
df_data = df.iloc[1:].copy().reset_index(drop=True)

# features and label
X = df_data.drop(columns=["dead-end", "source"])
y = df_data["dead-end"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Features shape:", df_features.shape)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Train label counts:")
print(y_train.value_counts())

print("Test label counts:")
print(y_test.value_counts())

Features shape: (51,)
X_train shape: (343, 51)
X_test shape: (86, 51)
y_train shape: (343,)
y_test shape: (86,)
Train label counts:
dead-end
0    284
1     59
Name: count, dtype: int64
Test label counts:
dead-end
0    71
1    15
Name: count, dtype: int64


## Step 3: Baseline Classifier

Before using ASP, we will train a standard ML classifier on the same train/test split
and evaluate it with the same metrics. This gives us a concrete reference point.

We are building a classifier that a planner will use to prune states from search.
Before looking at any numbers, consider what each type of prediction outcome costs the planner.

**Your task:** fill in the **Effect on planner** column for each row.

| Outcome | Classifier says | State actually is | Effect on planner |
|---|---|---|---|
| **True positive (TP)** | dead-end | dead-end | The planner perfectly prunes this state to avoid wasting time on searching irrelevant path towards goal state. |
| **True negative (TN)** | solvable | solvable | The planner keeps this outcome state in search queue, allowing it to continue searching a valid path towrads goal state.|
| **False positive (FP)** | dead-end | solvable | The planner inaccurately reports a solvable states into dead-end state. Hence, accuracy is hampered by misreporting a solvable state into a dead-end state. |
| **False negative (FN)** | solvable | dead-end | The planner waste compute time and memory searching/solving for a dead-end state. therefore, inefficient effect is posed.  |

Two metrics we will track throughout the notebook:

**Accuracy** = fraction of all predictions that are correct:
$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision** = fraction of dead-end predictions that are actually dead-end:
$$\text{precision} = \frac{TP}{TP + FP}$$

**Given the costs in the table above, which metric is more important for a planner that uses this classifier to prune states?**

**Your answer: Precision**

**Why?**

**Your answer: Well, Precision captures the fraction of dead-end state predictions that are actually dead-end states. Higher the precision means lower the number of false positives. Therefore, it is highly important for AI planner to curtail the misreporting of solvable state into a dead-end state. If this emerges in a critical path, the planner will completely fail to find a valid solution. Hence, making entire system unrobust.**


In [5]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

baseline = KNeighborsClassifier(n_neighbors=5)
baseline.fit(X_train, y_train)
y_hat_baseline = baseline.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_hat_baseline).ravel()

print(f"Baseline KNN, k=5 (random split)")
print(f"TP: {tp}  TN: {tn}  FP: {fp}  FN: {fn}")
print(f"accuracy:  {accuracy_score(y_test, y_hat_baseline):.4f}")
print(f"precision: {precision_score(y_test, y_hat_baseline, zero_division=0):.4f}")
print(f"recall:    {recall_score(y_test, y_hat_baseline, zero_division=0):.4f}")
print(f"f1:        {f1_score(y_test, y_hat_baseline, zero_division=0):.4f}")


Baseline KNN, k=5 (random split)
TP: 7  TN: 68  FP: 3  FN: 8
accuracy:  0.8721
precision: 0.7000
recall:    0.4667
f1:        0.5600


**Discussion questions:**

1. What does a **false positive** mean for a planner that uses this classifier to prune
   states from search? What does a **false negative** mean? Which is more dangerous, and
   why?

2. Given the precision you observed above, would you trust this classifier to prune states
   safely during planning?

**Your answer:**


In [5]:
# Sequence split: first 80% of rows in CSV order as training, last 20% as test.
# No shuffle, no stratify — we preserve the original ordering of the data.
df_data_ordered = df.iloc[1:].copy().reset_index(drop=True)
X_ordered = df_data_ordered.drop(columns=["dead-end", "source"])
y_ordered = df_data_ordered["dead-end"]

split_idx = int(len(X_ordered) * 0.8)
X_train_seq = X_ordered.iloc[:split_idx]
X_test_seq  = X_ordered.iloc[split_idx:]
y_train_seq = y_ordered.iloc[:split_idx]
y_test_seq  = y_ordered.iloc[split_idx:]

baseline_seq = KNeighborsClassifier(n_neighbors=5)
baseline_seq.fit(X_train_seq, y_train_seq)
y_hat_seq = baseline_seq.predict(X_test_seq)

# labels=[0,1] forces a 2x2 matrix even if one class is absent from the test set
tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_test_seq, y_hat_seq, labels=[0, 1]).ravel()

print(f"Baseline KNN, k=5 (sequence split)")
print(f"TP: {tp_s}  TN: {tn_s}  FP: {fp_s}  FN: {fn_s}")
print(f"accuracy:  {accuracy_score(y_test_seq, y_hat_seq):.4f}")
print(f"precision: {precision_score(y_test_seq, y_hat_seq, zero_division=0):.4f}")
print(f"recall:    {recall_score(y_test_seq, y_hat_seq, zero_division=0):.4f}")
print(f"f1:        {f1_score(y_test_seq, y_hat_seq, zero_division=0):.4f}")


Baseline KNN, k=5 (sequence split)
TP: 0  TN: 82  FP: 4  FN: 0
accuracy:  0.9535
precision: 0.0000
recall:    0.0000
f1:        0.0000


**Discussion:** Does the classifier still generalise when trained on earlier states and
tested on later ones? What might the change in performance tell you about the ordering
structure of the data?

**Your answer:**


## Step 4: Learning a Perfect Classifier using Answer Set Programming (ASP)

The clingo documentation is available in PDF format [here](https://wp.doc.ic.ac.uk/arusso/wp-content/uploads/sites/47/2015/01/clingo_guide.pdf).

The Python API documentation is available online [here](https://potassco.org/clingo/python-api/5.4/).

In [ ]:
from clingo import Control, Number, Function, Symbol
from sklearn.metrics import accuracy_score, recall_score, precision_score


class Feature:
    """
    A feature is defined by an index, complexity, and description.
    The index refers to a column in the dataset.
    The complexity measures the number of grammar rules needed to compute the feature.
    The description is a human-readable string describing the feature.
    """
    def __init__(self, index : int, complexity : int, description : str):
        self.index = index
        self.complexity = complexity
        self.description = description

    def __str__(self):
        return self.description

    def __repr__(self):
        return (
            f"Feature(index={self.index}, "
            f"complexity={self.complexity}, "
            f"description={self.description!r})"
        )

class TPerfectModel:
    """ A TPerfectModel is is a conjunction of features. 
    """
    def __init__(self, features : list[Feature]):
        self.features = features

    def predict(self, X : pd.DataFrame) -> pd.Series:
        """ Predict the label for each state in X. 
            The prediction is 1 (unsolvable) if any of the selected features has a value of 1, and 0 (solvable) otherwise.
        """
        return X.iloc[:, [feature.index for feature in self.features]].any(axis=1).astype(int)
    
    def __str__(self):
        if not self.features:
            return "TPerfectModel(features=[])"

        feature_lines = "\n".join(f"    {str(f)}" for f in self.features)
        return f"TPerfectModel(\n  features=[\n{feature_lines}\n]\n)"

    def __repr__(self):
        if not self.features:
            return "TPerfectModel(features=[])"

        feature_lines = ",\n".join(f"    {repr(f)}" for f in self.features)
        return f"TPerfectModel(\n  features=[\n{feature_lines}\n])"
    
    @staticmethod
    def parse_from_answer_set(symbols : list[Symbol], df_features : pd.Series) -> "TPerfectModel":
        """ Parse the selected features from the answer set symbols and update the model accordingly. 
        """
        selected_feature_indices = [symbol.arguments[0].number for symbol in symbols if symbol.name == "select"]
        features = [Feature(index=feature_index, complexity=df_features.iloc[feature_index], description=df_features.index[feature_index]) for feature_index in selected_feature_indices]
        return TPerfectModel(features)


def learn_t_perfect_model(X_train : pd.DataFrame, y_train : pd.Series, df_features : pd.Series) -> TPerfectModel:
    control = Control()

    control.add("solvable", ["s"], "solvable(s).")
    control.add("unsolvable", ["s"], "unsolvable(s).")
    control.add("feature", ["f", "c"], "feature(f, c).")
    control.add("value", ["f", "s", "v"], "value(f, s, v).")

    facts = []
    for feature_index, (_, feature_complexity) in enumerate(df_features.items()):
        facts.append(("feature", [Number(feature_index), Number(feature_complexity)]))
    for state_index, row in X_train.iterrows():
        label = y_train.loc[state_index]
        if label == 1:
            facts.append(("unsolvable", [Number(state_index)]))
        else:
            facts.append(("solvable", [Number(state_index)]))
        for feature_index, (feature_name, feature_state_value) in enumerate(row.items()):
            facts.append(("value", [Number(feature_index), Number(state_index), Number(feature_state_value)]))

    control.ground(facts)

    # TODO 1 — Choice rule.
    # Allow any subset of features to be selected.
    feature_selection = ""  # <-- your code

    # TODO 2 — Separation constraint.
    # Forbid models in which some unsolvable state and some solvable state are
    # indistinguishable under the selected features.
    separation_constraint = ""  # <-- your code

    # TODO 3 — Optimization objective.
    # Prefer models with smaller total complexity over the selected features.
    minimize_complexity = ""  # <-- your code

    # TODO 4 — Show statement.
    # Expose only the atoms that record which features were selected.
    show_selected = ""  # <-- your code

    control.add("base", [], f"""
        {feature_selection}

        {separation_constraint}

        {minimize_complexity}

        {show_selected}
    """)

    control.ground([("base", [])])

    control.configuration.solve.opt_mode = "optN"
    control.configuration.solve.models = 1

    with control.solve(yield_=True) as handle:
        for m in handle:
            pass
        result = handle.get()
        final_model = handle.last()

    print("result.satisfiable:", result.satisfiable)
    print("result.exhausted:", result.exhausted)
    print("result.interrupted:", result.interrupted)

    if final_model is not None:
        print("final cost:", final_model.cost)
        print("final optimality_proven:", final_model.optimality_proven)
        print("final model:")
        return TPerfectModel.parse_from_answer_set(final_model.symbols(shown=True), df_features)
    
    return None


In [ ]:
# Call the learning function to learn a TPerfectModel from the training data and print the resulting model.

t_perfect_model = learn_t_perfect_model(X_train, y_train, df_features)
print(t_perfect_model)

y_hat = t_perfect_model.predict(X_test)

accuracy = accuracy_score(y_test, y_hat)
recall = recall_score(y_test, y_hat)
precision = precision_score(y_test, y_hat)

print(f"accuracy: {accuracy:.4f}")
print(f"recall:   {recall:.4f}")
print(f"precision: {precision:.4f}")

### Checking Your Hypothesis

Compare the feature(s) selected by the T-Perfect model above against the predictions
you recorded in Step 1.

- Did the solver pick the features you expected?
- If the selected features differ from your prediction, what property of the selected
  feature makes it more useful for separating dead-end states than the one you chose?

**Your answer:**


## Step 5: Learning a T-Safe Classifier using ASP

We now change the objective slightly to learn T-Safe generalized unsolvability heuristics.

The change is minor, meaning we can reuse most of the code from above.

In [ ]:
from clingo import Control, Number, Function, Symbol
from sklearn.metrics import accuracy_score, recall_score, precision_score


class TSafeModel:
    """
    T-SAFE classifier represented as a disjunction over per-state T-perfect terms.
    A state is predicted unsolvable (1) if any term predicts unsolvable.
    """
    def __init__(self, terms: list[TPerfectModel]):
        self.terms = terms

    def predict(self, X: pd.DataFrame) -> pd.Series:
        # === Empty-disjunction case ===
        # If there are no terms, predict all states as solvable (return all zeros).
        # TODO: your code here

        # === Logical OR over terms ===
        # A state is predicted dead-end if ANY term in self.terms predicts it as
        # dead-end. Combine term predictions using a logical OR.
        # TODO: your code here

    def __str__(self):
        if not self.terms:
            return "TSafeModel(terms=[])"
        term_lines = "\n  ".join(str(t) for t in self.terms)
        return f"TSafeModel(\n  {term_lines}\n)"


def learn_tsafe_model(
    X_train: pd.DataFrame, y_train: pd.Series, df_features: pd.Series
) -> TSafeModel:
    """
    Learn a T-SAFE disjunction of T-perfect terms.
    For each unsolvable state, create an iteration dataset with:
    - all solvable states, and
    - exactly this one unsolvable state.
    Then learn one T-perfect term and OR it into the final model.
    """
    terms: list[TPerfectModel] = []
    unsolvable_rows = X_train[y_train == 1]

    print(f"Total unsolvable states to process: {len(unsolvable_rows)}")

    for i, (s_idx, s_row) in enumerate(unsolvable_rows.iterrows(), start=1):
        # === Skip already-covered states ===
        # If the current disjunction already predicts state s_idx as dead-end,
        # skip it — no new term is needed.
        # TODO: your code here

        # === Construct the per-state subproblem ===
        # Build a dataset containing all solvable training states plus only this
        # one unsolvable state. This is the input for one T-Perfect subproblem.
        X_iter = None  # TODO: your code
        y_iter = None  # TODO: your code

        print(f"  [{i}/{len(unsolvable_rows)}] state {s_idx}: solving ...", end=" ")
        term = learn_t_perfect_model(X_iter, y_iter, df_features)
        if term is None:
            print("no separating term (features not expressive enough)")
            continue

        # === Add term to the disjunction ===
        # If a term was found, add it to the disjunction.
        terms.append(term)
        print(f"term found → {term}")

    print(f"\nT-SAFE model learned with {len(terms)} term(s).")
    return TSafeModel(terms)


In [ ]:
# Call the learning function to learn a TSafeModel from the training data and print the resulting model.

tsafe_model = learn_tsafe_model(X_train, y_train, df_features)
print(tsafe_model)

y_hat = tsafe_model.predict(X_test)

accuracy = accuracy_score(y_test, y_hat)
recall = recall_score(y_test, y_hat)
precision = precision_score(y_test, y_hat)

print(f"accuracy: {accuracy:.4f}")
print(f"recall:   {recall:.4f}")
print(f"precision: {precision:.4f}")

## Step 6: Bake-Off Against ML Baselines

*(Open-ended capstone, intended for fast finishers and Day 2 discussion.)*

You will now compare your ASP classifiers against a range of standard ML approaches
using the **same train/test split**. The primary comparison axis is **precision and
false-positive rate**, not accuracy. Record which classifiers achieve zero false
positives and what that costs in recall.

**Required baseline: Decision Tree.** A decision tree learns a *disjunction of
conjunctions of feature tests*: each path from root to a positive leaf is a conjunction,
and the union of dead-end leaves is a disjunction. That is the same hypothesis space as
your T-Safe ASP model. Comparing the two surfaces what the ASP solver actually
contributes: a global search over feature subsets, plus a hard zero-false-positive
constraint on training data, in place of the tree's greedy entropy splits with no
precision guarantee. You may add other classifiers (KNN, GBM, MLP, anything you like)
on top of the required tree.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_hat_dt = dt.predict(X_test)

print(f"Decision Tree (default settings)")
print(f"accuracy:  {accuracy_score(y_test, y_hat_dt):.4f}")
print(f"recall:    {recall_score(y_test, y_hat_dt):.4f}")
print(f"precision: {precision_score(y_test, y_hat_dt):.4f}")

y_hat_dt_train = dt.predict(X_train)
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_train, y_hat_dt_train, labels=[0, 1]).ravel()
print(f"\nTraining set: TP={tp_t}  TN={tn_t}  FP={fp_t}  FN={fn_t}")

# Structural comparison with the ASP feature set.
print(f"\nTree depth: {dt.get_depth()}   leaves: {dt.get_n_leaves()}")
used_indices = sorted({int(i) for i in dt.tree_.feature if i >= 0})
print(f"Features used by the tree ({len(used_indices)}):")
for fi in used_indices:
    print(f"  [f{fi}] {X_train.columns[fi]}")

# Print the learned tree using short aliases (f0, f1, ...) so the output stays readable;
# cross-reference indices to the column names listed above.
feature_aliases = [f"f{i}" for i in range(X_train.shape[1])]
print("\nLearned tree:")
print(export_text(dt, feature_names=feature_aliases))


In [ ]:
import pandas as pd

results = []
for name, y_pred in [
    ("Baseline KNN (k=5)",  y_hat_baseline),
    ("T-Perfect ASP",       t_perfect_model.predict(X_test)),
    ("T-Safe ASP",          tsafe_model.predict(X_test)),
    ("Decision Tree",       y_hat_dt),
]:
    results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
    })

pd.DataFrame(results).set_index("Model")


**Discussion questions:**

1. Which classifiers achieve **zero false positives** on the test set? What is the cost of
   that guarantee in terms of recall or model complexity?

2. If you were deploying one of these classifiers to prune states during planning, which
   would you choose and under what conditions would you switch?

3. The T-Safe classifier is built one unsolvable state at a time from training data. What
   do you expect to happen to its false-positive rate and recall as the training set grows
   larger?

4. Look at the features the **decision tree** split on and the features **T-Perfect** and
   **T-Safe** selected. Are they the same set? If not, what does each model see that the
   other does not, and why might that be?

5. The decision tree and T-Safe both express a *disjunction of conjunctions of feature
   tests*: same hypothesis space, different search procedure. Compare them on
   (a) training-set false positives, (b) number of features used, and (c) the syntactic
   complexity of the chosen feature(s). What does the ASP encoding buy you that the
   greedy splitter does not, and what does it cost?

6. Decision-tree and decision-list learners have a long history of being used to learn
   control knowledge for classical planning (e.g., Yoon, Fern & Givan; Khardon; Martin
   & Geffner). Given what you just observed, what does the ASP-based approach add to that
   lineage, and where might it fall short?

**Your answer:**
